# MSD RecSys — pipeline notebook

End-to-end driver for the Million Song Dataset two-stage recommender.
The library lives in `msd_recsys/`; this notebook just stitches the stages together
and prints diagnostics between them.

**Stages**
1. Setup (install, mount Drive, set paths)
2. Load + filter + split data
3. Stage 1 — retrieval (ALS + semantic IDs -> hybrid pool)
4. Stage 2 — features + ranker
5. Evaluation (overall + bucketed)
6. Submission


## 1. Setup


In [1]:
# --- Detect environment ---
import sys, os
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

# --- Colab: clone (once), install the package, mount Drive ---
if IN_COLAB:
    if not os.path.exists('/content/ecs172-recommender-systems'):
        !git clone https://github.com/YOUR_USER/ecs172-recommender-systems.git /content/ecs172-recommender-systems
    %cd /content/ecs172-recommender-systems
    !pip install -q -e ./final_project
    from google.colab import drive
    drive.mount('/content/drive')
    print('running in Colab')

# --- Local: walk up from CWD to find the dir containing msd_recsys/, add to sys.path ---
# This lets `import msd_recsys` work even without `pip install -e .` — useful for fast iteration.
else:
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / 'msd_recsys').is_dir():
            if str(candidate) not in sys.path:
                sys.path.insert(0, str(candidate))
            print(f'running locally; added {candidate} to sys.path')
            break
    else:
        raise RuntimeError(
            'Could not find msd_recsys/ above the notebook. '
            'Are you sure the notebook is inside final_project/notebooks/?'
        )


In [4]:
from pathlib import Path
import numpy as np
import pandas as pd

from msd_recsys import data, retrieval, features, ranker, eval as ev, diagnostics as diag
from msd_recsys.checkpoint import checkpoint, set_checkpoint_dir, list_checkpoints

# Adjust these for your environment.
if IN_COLAB:
    DATA_DIR = Path('/content/drive/MyDrive/msd_data')
    CKPT_DIR = Path('/content/drive/MyDrive/msd_checkpoints')
else:
    DATA_DIR = Path('./data')
    CKPT_DIR = Path('./checkpoints')

CKPT_DIR.mkdir(parents=True, exist_ok=True)
set_checkpoint_dir(CKPT_DIR)
print(f"DATA_DIR = {DATA_DIR}")
print(f"CKPT_DIR = {CKPT_DIR}")
print(f"existing checkpoints: {list_checkpoints()}")


## 2. Load + filter + split


In [ ]:
# --- Load raw interactions + metadata (cached) ---
interactions = checkpoint(
    "interactions_raw_v1",
    lambda: data.load_triplets(DATA_DIR / "train_triplets.txt"),
)
metadata = checkpoint(
    "metadata_v1",
    lambda: data.load_track_metadata(DATA_DIR / "track_metadata.db"),
)
print(f"interactions: {interactions.shape}")
print(f"metadata:     {metadata.shape}")
diag.diag_data(interactions, items_catalog=metadata)


In [ ]:
# --- Aggressive filter: drop tail items and low-activity users ---
MIN_SONG_LISTENS = 50
MIN_USER_LISTENS = 20

filtered = checkpoint(
    f"interactions_filtered_s{MIN_SONG_LISTENS}_u{MIN_USER_LISTENS}",
    lambda: data.filter_interactions(
        interactions,
        min_song_listens=MIN_SONG_LISTENS,
        min_user_listens=MIN_USER_LISTENS,
    ),
)
diag.diag_filter(interactions, filtered)


In [ ]:
# --- Hold-out split mirroring the MSD challenge format ---
KEEP_LAST_N = 5
train_inner, valid = checkpoint(
    f"split_n{KEEP_LAST_N}",
    lambda: data.holdout_split(filtered, n_per_user=KEEP_LAST_N),
)
print(f"train_inner: {len(train_inner):,} rows, {train_inner.user_id.nunique():,} users")
print(f"valid      : {len(valid):,} rows,    {valid.user_id.nunique():,} users")


In [ ]:
# --- Build sparse user-item matrix for ALS ---
ui_matrix, user_to_ix, item_to_ix = checkpoint(
    "ui_matrix_v1",
    lambda: data.build_user_item_matrix(train_inner, confidence_alpha=40.0),
)
print(f"user-item matrix: {ui_matrix.shape}, nnz={ui_matrix.nnz:,}")


## 3. Stage 1 — Retrieval


In [ ]:
# --- ALS via implicit (GPU-accelerated on A100 if available) ---
als = checkpoint(
    "als_f64_r0.01_i30",
    lambda: retrieval.ALSRetriever(factors=64, regularization=0.01, iterations=30).fit(
        ui_matrix, user_to_ix, item_to_ix,
    ),
)

# Optional sanity check — pass a few well-known song titles in if you have them.
# Build a title lookup from metadata for prettier diagnostic output.
title_by_song = dict(zip(metadata.song_id, metadata.title))
diag.diag_als(als, item_titles=title_by_song)


In [ ]:
# --- Semantic-ID retriever (metadata-only for now; revisit if adding audio) ---
# Filter metadata to the items that appear in our filtered training set so the
# semantic-ID space matches what we recommend over.
items_in_train = set(train_inner.song_id.unique())
metadata_in_train = metadata[metadata.song_id.isin(items_in_train)].copy()

sid = checkpoint(
    "sid_metadata_v1",
    lambda: retrieval.SemanticIDRetriever().fit(metadata_in_train),
)
diag.diag_semantic_codes(sid, item_titles=title_by_song, n_show=8)


In [ ]:
# --- Retrieve candidates for validation users ---
TOPK_ALS = 1000
TOPK_SID = 1000

valid_users = sorted(valid.user_id.unique())
# Only users present in train_inner can be scored by ALS
valid_users = [u for u in valid_users if u in user_to_ix]
valid_user_indices = np.array([user_to_ix[u] for u in valid_users])

hist_by_user = data.histories_from_df(train_inner)
valid_histories = [hist_by_user.get(u, []) for u in valid_users]
valid_truth = valid.groupby("user_id")["song_id"].apply(set).to_dict()

als_ids, als_sc = checkpoint(
    f"als_topk_v_{TOPK_ALS}",
    lambda: als.recommend_batch(ui_matrix, top_k=TOPK_ALS, user_indices=valid_user_indices),
)
sid_ids, sid_sc = checkpoint(
    f"sid_topk_v_{TOPK_SID}",
    lambda: sid.recommend_for_histories(
        valid_histories, top_k=TOPK_SID,
        already_owned=[set(h) for h in valid_histories],
    ),
)
valid_cands = retrieval.build_candidate_pool(als_ids, als_sc, sid_ids, sid_sc)
diag.diag_pool(valid_cands, truth_by_user=valid_truth, user_ids=valid_users)


## 4. Stage 2 — Features + ranker


In [ ]:
# --- Build item feature table (one-shot, used by every (user, candidate) row) ---
item_pop = train_inner.groupby("song_id").size()
item_table = features.ItemFeatureTable.build(
    metadata=metadata_in_train,
    item_popularity=item_pop,
    semantic_codes=sid.item_codes,
)
print(f"item feature table: {len(item_table.item_ids):,} items")


In [ ]:
# --- 80/20 split of validation users for ranker fit vs eval ---
rng = np.random.default_rng(42)
perm = rng.permutation(len(valid_users))
split = int(0.8 * len(valid_users))
fit_ix, eval_ix = perm[:split], perm[split:]

def slice_(arr, ixs): return [arr[i] for i in ixs]
fit_users  = slice_(valid_users,     fit_ix)
fit_hist   = slice_(valid_histories, fit_ix)
fit_cands  = slice_(valid_cands,     fit_ix)
eval_users = slice_(valid_users,     eval_ix)
eval_hist  = slice_(valid_histories, eval_ix)
eval_cands = slice_(valid_cands,     eval_ix)

X_fit,  y_fit,  k_fit,  g_fit  = features.build_feature_rows(
    fit_users, fit_hist, fit_cands, item_table, truth_by_user=valid_truth,
)
X_eval, y_eval, k_eval, g_eval = features.build_feature_rows(
    eval_users, eval_hist, eval_cands, item_table, truth_by_user=valid_truth,
)

# Drop zero-sized groups (users whose candidate dicts came back empty)
g_fit_nz  = [g for g in g_fit  if g > 0]
g_eval_nz = [g for g in g_eval if g > 0]
print(f"ranker fit : X={X_fit.shape}, positives={int(y_fit.sum()):,} ({y_fit.mean():.2%})")
print(f"ranker eval: X={X_eval.shape}, positives={int(y_eval.sum()):,} ({y_eval.mean():.2%})")
diag.diag_features(X_fit, y_fit, features.FEATURE_NAMES)


In [ ]:
# --- Train LightGBM lambdarank ---
model = ranker.train_lambdarank(
    X_fit, y_fit, group_sizes=g_fit_nz,
    eval_set=(X_eval, y_eval), eval_group=g_eval_nz,
    n_estimators=300, max_position=500, early_stopping_rounds=20,
)
diag.diag_permutation_importance(model, X_eval, y_eval, features.FEATURE_NAMES, rng=rng)


## 5. Evaluation


In [ ]:
# --- Rank candidates -> top-500 per user, compute metrics ---
eval_owned = {u: set(h) for u, h in zip(eval_users, eval_hist)}
ranked = ranker.rank_candidates(model, X_eval, k_eval, top_k=500, exclude_owned=eval_owned)

map500    = ev.map_at_k(ranked,    {u: valid_truth[u] for u in eval_users if u in valid_truth}, k=500)
recall500 = ev.mean_recall_at_k(ranked, {u: valid_truth[u] for u in eval_users if u in valid_truth}, k=500)
ndcg500   = ev.mean_ndcg_at_k(ranked,   {u: valid_truth[u] for u in eval_users if u in valid_truth}, k=500)

# Retrieval ceiling on the eval slice
ret_recall = ev.retrieval_recall(
    {u: set(c.keys()) for u, c in zip(eval_users, eval_cands)},
    {u: valid_truth[u] for u in eval_users if u in valid_truth},
)

# Sum-of-scores baseline
baseline_scores = X_eval[:, features.FEATURE_NAMES.index("als_score")] \
                + X_eval[:, features.FEATURE_NAMES.index("semantic_score")]
baseline_ranked: dict = {}
for (u, it), s in zip(k_eval, baseline_scores):
    baseline_ranked.setdefault(u, []).append((it, float(s)))
for u, lst in baseline_ranked.items():
    lst.sort(key=lambda x: -x[1])
    owned = eval_owned.get(u, set())
    baseline_ranked[u] = [it for it, _ in lst if it not in owned][:500]
baseline_map500 = ev.map_at_k(baseline_ranked, {u: valid_truth[u] for u in eval_users if u in valid_truth}, k=500)

print(f"MAP@500     : {map500:.4f}")
print(f"Recall@500  : {recall500:.4f}")
print(f"NDCG@500    : {ndcg500:.4f}")
print(f"baseline MAP@500 (als+sid sum): {baseline_map500:.4f}")
diag.diag_results(map_score=map500, retrieval_recall=ret_recall, baseline_map=baseline_map500)


In [ ]:
# --- Bucketed evaluation: by user history size ---
user_hist_size = {u: len(h) for u, h in zip(eval_users, eval_hist)}
def history_bucket(n):
    if n < 30:   return "0_light"     # leading underscores for sort order
    if n < 100:  return "1_medium"
    return "2_heavy"
user_buckets = {u: history_bucket(n) for u, n in user_hist_size.items()}

bucketed = ev.metrics_by_bucket(ranked, {u: valid_truth[u] for u in eval_users}, user_buckets, k=500)
print("by user-history bucket:")
print(bucketed.to_string(index=False))


In [ ]:
# --- Catalog coverage + tail-focused MAP ---
catalog_size = len(item_table.item_ids)
coverage = ev.catalog_coverage(ranked, catalog_size, k=500)
print(f"catalog coverage @500: {coverage:.2%}  ({int(coverage * catalog_size):,} of {catalog_size:,} items recommended to >=1 user)")

# Popularity tiers (head/torso/tail by cumulative listen share)
item_tier = ev.popularity_tiers(item_pop)
tail_map = ev.tail_focused_map(ranked, {u: valid_truth[u] for u in eval_users}, item_tier, k=500, tier="tail")
torso_map = ev.tail_focused_map(ranked, {u: valid_truth[u] for u in eval_users}, item_tier, k=500, tier="torso")
head_map = ev.tail_focused_map(ranked, {u: valid_truth[u] for u in eval_users}, item_tier, k=500, tier="head")
print(f"\nMAP@500 by ground-truth popularity tier:")
print(f"  head : {head_map:.4f}")
print(f"  torso: {torso_map:.4f}")
print(f"  tail : {tail_map:.4f}")
print("  -> the tail number is the 'discovery' signal — does the system help users find long-tail songs they actually played?")


## 6. Submission


In [ ]:
# --- Retrain Stage 1 on full filtered data + rank for test users ---
# Skeleton — adapt once you've confirmed which hidden test file to evaluate against.

# 1. Rebuild ui_matrix on `filtered` (not train_inner), refit ALS, refit SID.
# 2. Load test user_ids from kaggle_users.txt or the canonical hidden file.
# 3. For each test user, retrieve TOPK_ALS + TOPK_SID, build features, rank to top-500.
# 4. Format CSV per the MSD challenge format and validate.

# Placeholder — uncomment and adapt:
# test_users = data.load_user_list(DATA_DIR / "kaggle_users.txt")
# ... retrieve + rank ...
# submission.to_csv('submission.csv', index=False)
print("submission cell is a skeleton — fill in once test format is confirmed.")
